# Ingestão da Camada Bronze

Notebook para leitura dos dados da camada **Staging** e carga na camada **Bronze** do Data Lakehouse.
Os dados são copiados da staging com adição de metadados de rastreabilidade do processamento.
As transformações para Silver e Gold serão feitas em notebooks subsequentes.


## 📚 Sobre este Notebook

Este notebook implementa a ingestão da **camada Bronze** a partir dos dados já carregados na **camada Staging**.

### Arquitetura Medalhão

* **📂 Staging**: Dados brutos copiados diretamente do Google Drive (notebook MVP02-Download)
* **🥉 Bronze**: Dados brutos com metadados de rastreabilidade do processamento (este notebook)
* **🥈 Silver**: Dados limpos e padronizados  (notebook MVP04-Silver)
* **🥇 Gold**: Dados agregados e prontos para consumo  (notebook MVP05-Gold)

### 🎯 Objetivo
Ler as tabelas da camada Staging e carregá-las na camada Bronze, adicionando metadados de rastreabilidade:
* `data_carga_bronze`: timestamp de quando os dados foram processados para Bronze
* `origem_tabela`: identifica a tabela de staging de origem

In [0]:
%sql
-- Define o catálogo ativo como mvpDataEngineer no Unity Catalog.
USE CATALOG mvpDataEngineer;

-- Define o schema ativo como bronze dentro do catálogo atual.
USE SCHEMA bronze;

## 🥉 Etapa 1: Ingestão da Staging para Bronze

Lê cada tabela da camada **Staging**, adiciona metadados de rastreabilidade do processamento e grava na camada **Bronze**.

### Metadados adicionados:
* `data_carga_bronze`: timestamp de quando os dados foram processados para Bronze
* `origem_tabela`: identifica a tabela de staging de origem (ex: `staging.leitura_pluviometros`)

### Princípios da camada Bronze:
* ✅ **Preservar dados brutos** - sem transformação de conteúdo
* ✅ **Adicionar rastreabilidade** - metadados de processamento
* ✅ **Formato Delta Lake** - formato otimizado do Databricks
* ✅ **Histórico completo** - possibilita auditoria e reprocessamento

In [0]:
# =========================================
# INGESTÃO DA STAGING PARA BRONZE
# =========================================
from pyspark.sql.functions import current_timestamp, lit

print("Iniciando carga da Staging para Bronze...\n")

# =========================================
# DEFINIR TABELAS DA STAGING
# =========================================
# Lista de tabelas da staging que serão processadas para bronze
tabelas_staging = [
    "leitura_pluviometros",
    "medicao_bacias",
    "medicao_laje"
]

print(f"{len(tabelas_staging)} tabela(s) serão processadas:\n")
print("="*80)

# =========================================
# PROCESSAR CADA TABELA
# =========================================
for tabela in tabelas_staging:
    print(f"\n Processando: {tabela}")
    
    # -----------------------------------------
    # PASSO 1: LER DA STAGING
    # -----------------------------------------
    # Lê a tabela da camada staging (dados brutos do Google Drive)
    df_staging = spark.table(f"mvpDataEngineer.staging.{tabela}")
    row_count = df_staging.count()
    print(f"  - Registros lidos da staging: {row_count}")
    print(f"  - Colunas originais: {len(df_staging.columns)}")
    
    # ------------------------------------------------
    # PASSO 2: ADICIONAR METADADOS DE RASTREABILIDADE
    # ------------------------------------------------
    # data_carga_bronze: quando os dados foram processados para Bronze
    # origem_tabela: identifica a tabela de origem
    df_bronze = df_staging \
        .withColumn("data_carga_bronze", current_timestamp()) \
        .withColumn("origem_tabela", lit(f"staging.{tabela}"))
    
    # -----------------------------------------
    # PASSO 3: SALVAR NA CAMADA BRONZE
    # -----------------------------------------
    # mode="overwrite": substitui dados existentes
    table_name = f"mvpDataEngineer.bronze.{tabela}"
    
    df_bronze.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
    
    print(f" -Tabela criada: {table_name}")
    print(f" -Registros salvos: {row_count}")
    print(f" -Total de colunas (com metadados bronze): {len(df_bronze.columns)}")

print("\n" + "="*80)
print("Ingestão Bronze concluída! Tabelas criadas:")
print("="*80)

# Visualizar estrutura das tabelas bronze criadas
display(spark.sql("SHOW TABLES IN mvpDataEngineer.bronze"))

## 🔍 Etapa 2: Validação dos Dados Bronze

#### O que verifica?
* **Total de registros** por tabela
* **Última carga** na camada Bronze
* **Amostras** dos dados carregados
* **Colunas de metadados** adicionadas

In [0]:
%sql
-- Resumo das tabelas Bronze: total de registros e última carga
SELECT 
  'LEITURA_PLUVIOMETROS' as tabela,
  COUNT(*) as total_registros,
  MAX(data_carga_bronze) as ultima_carga_bronze,
  MAX(data_ingestao) as ultima_ingestao_staging
FROM bronze.leitura_pluviometros

UNION ALL

SELECT 
  'MEDICAO_BACIAS' as tabela,
  COUNT(*) as total_registros,
  MAX(data_carga_bronze) as ultima_carga_bronze,
  MAX(data_ingestao) as ultima_ingestao_staging
FROM bronze.medicao_bacias

UNION ALL

SELECT 
  'MEDICAO_LAJE' as tabela,
  COUNT(*) as total_registros,
  MAX(data_carga_bronze) as ultima_carga_bronze,
  MAX(data_ingestao) as ultima_ingestao_staging
FROM bronze.medicao_laje

ORDER BY tabela;

In [0]:
%sql
-- Amostra de LEITURA_PLUVIOMETROS (primeiras 5 linhas)
-- Inclui os metadados de staging e bronze
SELECT * FROM mvpDataEngineer.bronze.leitura_pluviometros LIMIT 10;

In [0]:
%sql
-- Amostra de MEDICAO_BACIAS (primeiras 5 linhas)
SELECT * FROM mvpDataEngineer.bronze.medicao_bacias LIMIT 10;

In [0]:
%sql
-- Amostra de MEDICAO_LAJE (primeiras 5 linhas)
SELECT * FROM mvpDataEngineer.bronze.medicao_laje LIMIT 10;

## 📝 Etapa 3: Documentação das Tabelas e Colunas

### Por que documentar?
Comentários no Unity Catalog tornam os dados autodescritivos: qualquer pessoa que consultar as tabelas via Catalog Explorer, SQL ou dashboards verá a descrição de cada coluna sem precisar procurar em documentação externa.

### Como funciona?
Usamos `COMMENT ON TABLE` e `COMMENT ON COLUMN` para registrar descrições diretamente nos metadados do Unity Catalog. Os comentários ficam visíveis em:
* **Catalog Explorer** — descrição de tabelas e colunas
* **Autocomplete do SQL Editor** — tooltip ao digitar nomes de colunas
* **Lineage e Dashboards** — contexto para análise
* **APIs do Unity Catalog** — acesso programático à documentação

In [0]:
%sql
-- =========================================
-- COMENTÁRIOS DA TABELA: leitura_pluviometros
-- =========================================

COMMENT ON TABLE mvpDataEngineer.bronze.leitura_pluviometros IS 'Leituras pluviométricas diárias das estações de medição de chuva, provenientes do Google Drive via camada Staging';

COMMENT ON COLUMN mvpDataEngineer.bronze.leitura_pluviometros.PROTOCOLO IS 'Número do protocolo de solicitação';
COMMENT ON COLUMN mvpDataEngineer.bronze.leitura_pluviometros.SOLICDATA IS 'Data e hora da solicitação';
COMMENT ON COLUMN mvpDataEngineer.bronze.leitura_pluviometros.STATUS IS 'Status da solicitação';
COMMENT ON COLUMN mvpDataEngineer.bronze.leitura_pluviometros.UNIDADE IS 'Unidade federativa de origem (ex: RN)';
COMMENT ON COLUMN mvpDataEngineer.bronze.leitura_pluviometros.ESTACAO IS 'Nome da estação pluviométrica';
COMMENT ON COLUMN mvpDataEngineer.bronze.leitura_pluviometros.DATAREGISTRO IS 'Data do registro da leitura';
COMMENT ON COLUMN mvpDataEngineer.bronze.leitura_pluviometros.MILIMETROS IS 'Quantidade de chuva medida em milímetros';
COMMENT ON COLUMN mvpDataEngineer.bronze.leitura_pluviometros.OBSERVACOES IS 'Observações adicionais do registro';

-- Metadados de rastreabilidade (Staging)
COMMENT ON COLUMN mvpDataEngineer.bronze.leitura_pluviometros.data_ingestao IS 'Timestamp da ingestão na camada Staging';
COMMENT ON COLUMN mvpDataEngineer.bronze.leitura_pluviometros.origem IS 'Origem dos dados (google_drive)';
COMMENT ON COLUMN mvpDataEngineer.bronze.leitura_pluviometros.arquivo_origem IS 'Nome do arquivo Excel de origem';

-- Metadados de rastreabilidade (Bronze)
COMMENT ON COLUMN mvpDataEngineer.bronze.leitura_pluviometros.data_carga_bronze IS 'Timestamp da carga na camada Bronze';
COMMENT ON COLUMN mvpDataEngineer.bronze.leitura_pluviometros.origem_tabela IS 'Tabela de staging de origem';

In [0]:
%sql
-- =========================================
-- COMENTÁRIOS DA TABELA: medicao_bacias
-- =========================================

COMMENT ON TABLE mvpDataEngineer.bronze.medicao_bacias IS 'Medições das bacias de evaporadoras: mede a taxa de evaporação diária de água doce e salmoura, provenientes do Google Drive via camada Staging';

COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.PROTOCOLO IS 'Número do protocolo de solicitação';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.SOLICDATA IS 'Data e hora da solicitação';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.STATUS IS 'Status da solicitação';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.UNIDADE IS 'Unidade federativa de origem (ex: RN)';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.BACIA IS 'Identificador da bacia evaporadora';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.DATAREGISTRO IS 'Data do registro da medição';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.AGUADOCE_COLOCADO IS 'Volume de água doce colocada na bacia';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.AGUADOCE_RETIRADO IS 'Volume de água doce retirada da bacia';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.SALMOURA_COLOCADO IS 'Volume de salmoura colocada na bacia';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.SALMOURA_RETIRADO IS 'Volume de salmoura retirada da bacia';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.SALMOURA_DENSIDADE IS 'Densidade da salmoura (g/cm³)';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.OBSERVACOES IS 'Observações adicionais do registro';

-- Metadados de rastreabilidade (Staging)
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.data_ingestao IS 'Timestamp da ingestão na camada Staging';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.origem IS 'Origem dos dados (google_drive)';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.arquivo_origem IS 'Nome do arquivo Excel de origem';

-- Metadados de rastreabilidade (Bronze)
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.data_carga_bronze IS 'Timestamp da carga na camada Bronze';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_bacias.origem_tabela IS 'Tabela de staging de origem';

In [0]:
%sql
-- =========================================
-- COMENTÁRIOS DA TABELA: medicao_laje
-- =========================================

COMMENT ON TABLE mvpDataEngineer.bronze.medicao_laje IS 'Medições da laje dos cristalizadores: leitura mensal das réguas de nível por cristalizador, provenientes do Google Drive via camada Staging';

COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.PROTOCOLO IS 'Número do protocolo de solicitação';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.SOLICDATA IS 'Data e hora da solicitação';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.STATUS IS 'Status da solicitação';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.UNIDADE IS 'Unidade federativa de origem (ex: RN)';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.DATAMEDICAO IS 'Data da medição da laje';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.CRISTALIZADOR IS 'Número de identificação do cristalizador';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.AREAHECTARES IS 'Área do cristalizador em hectares';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.REGUA_1 IS 'Leitura da régua 1 de medição de nível';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.REGUA_2 IS 'Leitura da régua 2 de medição de nível';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.REGUA_3 IS 'Leitura da régua 3 de medição de nível';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.REGUA_4 IS 'Leitura da régua 4 de medição de nível';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.REGUA_5 IS 'Leitura da régua 5 de medição de nível';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.REGUA_6 IS 'Leitura da régua 6 de medição de nível';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.REGUA_7 IS 'Leitura da régua 7 de medição de nível';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.REGUA_8 IS 'Leitura da régua 8 de medição de nível';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.OBSERVACOES IS 'Observações adicionais do registro';

-- Metadados de rastreabilidade (Staging)
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.data_ingestao IS 'Timestamp da ingestão na camada Staging';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.origem IS 'Origem dos dados (google_drive)';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.arquivo_origem IS 'Nome do arquivo Excel de origem';

-- Metadados de rastreabilidade (Bronze)
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.data_carga_bronze IS 'Timestamp da carga na camada Bronze';
COMMENT ON COLUMN mvpDataEngineer.bronze.medicao_laje.origem_tabela IS 'Tabela de staging de origem';